# **Install and Import Libraries**

> ##### **Make sure the secrets.env file is in the config folder. An example for secrets.env can be found in config/secrets_example.env file**

In [25]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [26]:
from dotenv import load_dotenv
import os
import requests
from bs4 import BeautifulSoup
import time
import json
import re
from datetime import datetime
from neo4j import GraphDatabase

# load config
load_dotenv("../config/config.env")

# load secrets
load_dotenv("../config/secrets.env")

True

# **1. Fetch Course Links**

In [27]:
BASE_URL = "https://mi.malax.fi"
COURSES_URL = f"{BASE_URL}/kurser/"
REQUEST_TIMEOUT = 10

def fetch_course_links():
    try:
        course_links = []
        current_url = COURSES_URL
        
        while current_url:
            response = requests.get(current_url, timeout=REQUEST_TIMEOUT)
            response.raise_for_status()
            soup = BeautifulSoup(response.content, 'html.parser')
            
            # Fetch courses on current page
            page_courses = 0
            for a_tag in soup.find_all('a', class_='course-name'):
                href = a_tag.get('href')
                if href:
                    full_link = BASE_URL + href if href.startswith('/') else href
                    # avoid adding duplicates if the website acts weirdly
                    if not any(c['link'] == full_link for c in course_links):
                        course_links.append({
                            'title': a_tag.get_text(strip=True),
                            'link': full_link
                        })
                        page_courses += 1
            
            print(f"Scraped {page_courses} courses from {current_url}")
            
            # Look for "Nästa" (Next) button for pagination
            next_url = None
            next_btn = soup.find(string=lambda text: "Nästa" in text if text else False)
            if next_btn:
                parent_a = next_btn.find_parent('a')
                if parent_a and parent_a.get('href'):
                    href = parent_a.get('href')
                    next_url = BASE_URL + "/" + href if not href.startswith('/') else BASE_URL + href
                    # Sometimes href is just like 'kurser/showResults?start=20', make sure not to double down on 'kurser'
                    if not href.startswith('/') and not next_url.replace(BASE_URL + "/", "").startswith("kurser/"):
                           next_url = BASE_URL + "/kurser/" + href

            current_url = next_url
            if current_url:
                 time.sleep(1) # politely wait before next page fetch

        return course_links
    except Exception as e:
        print(f"Error fetching courses: {e}")
        return []

course_links = fetch_course_links()
print(f"Found {len(course_links)} courses.")
if course_links:
    print("\nFirst 3 courses:")
    for c in course_links[:3]:
        print(f"- {c['title']}: {c['link']}")

Scraped 20 courses from https://mi.malax.fi/kurser/
Scraped 20 courses from https://mi.malax.fi/kurser/showResults?start=20
Scraped 20 courses from https://mi.malax.fi/kurser/showResults?start=40
Scraped 20 courses from https://mi.malax.fi/kurser/showResults?start=60
Scraped 20 courses from https://mi.malax.fi/kurser/showResults?start=80
Scraped 20 courses from https://mi.malax.fi/kurser/showResults?start=100
Scraped 15 courses from https://mi.malax.fi/kurser/showResults?start=120
Found 135 courses.

First 3 courses:
- After work gymnastik: https://mi.malax.fi/kurser/visa/3220-830240-after-work-gymnastik
- Aktiv närkultur: https://mi.malax.fi/kurser/visa/2937-130100-aktiv-narkultur
- Aktuellt för pensionärer: https://mi.malax.fi/kurser/visa/3128-320204-aktuellt-for-pensionarer


# **2. Scrape Complete Details from Course Pages**

In [28]:
def scrape_course_details(url):
    try:
        response = requests.get(url, timeout=REQUEST_TIMEOUT)
        response.raise_for_status()
        soup = BeautifulSoup(response.content, 'html.parser')
        
        details = {}
        
        # Helper to extract data from a specific div class
        def extract_field(class_name, prefix_strip=None):
            div = soup.find('div', class_=lambda c: c and class_name in c)
            if div:
                # Remove strong tags text if matching prefix to clean it up
                text = div.get_text(" ", strip=True)
                if prefix_strip and text.startswith(prefix_strip):
                    text = text[len(prefix_strip):].strip()
                return text
            return ""

        details['coursecode'] = extract_field('course-page-coursecode', prefix_strip="Kurskod")
        details['price'] = extract_field('course-page-price', prefix_strip="Kursavgift")
        details['teacher'] = extract_field('course-page-teachers', prefix_strip="Lärare")
        details['location'] = extract_field('course-location', prefix_strip="Plats")
        
        times_raw = extract_field('course-times', prefix_strip="Tidpunkt")
        # Extract start and end dates from times string (e.g., "12.01.2026 - 13.04.2026")
        details['start_date'] = None
        details['end_date'] = None
        
        date_match = re.search(r"(\d{2}\.\d{2}\.\d{4})\s*-\s*(\d{2}\.\d{2}\.\d{4})", times_raw)
        if date_match:
            try:
                start_d = datetime.strptime(date_match.group(1), "%d.%m.%Y").strftime("%Y-%m-%d")
                end_d = datetime.strptime(date_match.group(2), "%d.%m.%Y").strftime("%Y-%m-%d")
                details['start_date'] = start_d
                details['end_date'] = end_d
                # Remove the date range from the times text to just keep schedule/lessons info
                times_raw = times_raw.replace(date_match.group(0), "").strip()
                if times_raw.startswith("|"): 
                    times_raw = times_raw[1:].strip()
            except ValueError:
                pass
                
        day_time_match = re.search(r'([A-Za-zåäöÅÄÖ]+\s+\d{2}:\d{2}\s*-\s*\d{2}:\d{2})', times_raw)
        if day_time_match:
            times_raw = day_time_match.group(1).strip()
        else:
            for cutoff in ['Lektioner:', 'Visa alla tidpunkter']:
                idx = times_raw.find(cutoff)
                if idx != -1:
                    times_raw = times_raw[:idx].strip()
                    
        details['times'] = times_raw
        
        signup_raw = extract_field('course-signup-times', prefix_strip="Anmälningstid")
        details['signup_start'] = None
        details['signup_end'] = None
        
        if signup_raw:
            parts = [p.strip() for p in signup_raw.split('-')]
            if len(parts) >= 1:
                try:
                    start_dt = datetime.strptime(parts[0], "%d.%m.%Y %H:%M")
                    details['signup_start'] = start_dt.strftime("%Y-%m-%dT%H:%M:%S")
                except ValueError:
                    pass
                    
            if len(parts) >= 2:
                if "non" in parts[1].lower() or "nonstop" in parts[1].lower():
                    if details.get('end_date'):
                        details['signup_end'] = f"{details['end_date']}T23:59:59"
                else:
                    try:
                        end_dt = datetime.strptime(parts[1], "%d.%m.%Y %H:%M")
                        details['signup_end'] = end_dt.strftime("%Y-%m-%dT%H:%M:%S")
                    except ValueError:
                        pass
                        
        details['signup_times'] = signup_raw
        
        details['spots_available'] = extract_field('course-page-spots', prefix_strip="Lediga platser")
        details['languages'] = extract_field('course-page-languages', prefix_strip="Kursspråk")
        details['description'] = extract_field('course-page-description')

        return details
    except Exception as e:
        print(f"Error scraping {url}: {e}")
        return None

# Use a slice to limit testing, e.g. course_links[:5] for faster run
# We map through our fetched link list
courses_data = []
for idx, course in enumerate(course_links): # process all courses
    print(f"[{idx+1}/{len(course_links)}] Scraping: {course['title']} ...", end="")
    details = scrape_course_details(course['link'])
    if details:
        full_course = {**course, **details}
        courses_data.append(full_course)
        print(" ✓")
    else:
        print(" ✗")
    time.sleep(1) # Be polite

if courses_data:
    print("\nSample Course Data:")
    print(json.dumps(courses_data[0], indent=2, ensure_ascii=False))

[1/135] Scraping: After work gymnastik ... ✓
[2/135] Scraping: Aktiv närkultur ... ✓
[3/135] Scraping: Aktuellt för pensionärer ... ✓
[4/135] Scraping: Aktuellt för pensionärer ... ✓
[5/135] Scraping: Aktuellt för pensionärer ... ✓
[6/135] Scraping: Animationsfabriken (åk 3-6) ... ✓
[7/135] Scraping: Att beskära äppelträd ... ✓
[8/135] Scraping: Bildkonstskolan, fortsättning och fördjupning ... ✓
[9/135] Scraping: Bildkonstskolan, grundkurs ... ✓
[10/135] Scraping: Bildkonstskolan, grundkurs ... ✓
[11/135] Scraping: Bugg inför sommaren ... ✓
[12/135] Scraping: Cirkelträning i gympasal ... ✓
[13/135] Scraping: Dans som motion ... ✓
[14/135] Scraping: Effektiv cirkelträning ... ✓
[15/135] Scraping: Effektiv cirkelträning - vår/sommar ... ✓
[16/135] Scraping: Español 2 ... ✓
[17/135] Scraping: FRK Första Hjälpen-kurs FHJ 1® (16 h), kombination: när + distanskurs ... ✓
[18/135] Scraping: Flitiga fingrar ... ✓
[19/135] Scraping: Frisk och skadefri löpare, workshop ... ✓
[20/135] Scraping: C

# **3. Connect to Neo4j Database**

In [29]:
# Initialize Neo4j driver
NEO4J_URI = os.getenv("NEO4J_URI")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))

# Test connection
try:
    with driver.session() as session:
        result = session.run("RETURN 1")
        print("✓ Successfully connected to Neo4j database")
except Exception as e:
    print(f"✗ Failed to connect to Neo4j: {e}")

✓ Successfully connected to Neo4j database


# **4. Create Course Nodes in Neo4j**

In [30]:
def create_or_update_course_node(driver, course):
    """
    Create or update a Course node in Neo4j.
    Uses the coursecode as a unique identifier.
    """
    with driver.session() as session:
        query = """
        MERGE (c:Course {coursecode: $coursecode})
        SET 
            c.title = $title,
            c.link = $link,
            c.price = $price,
            c.teacher = $teacher,
            c.location = $location,
            c.start_date = CASE WHEN $start_date IS NOT NULL THEN date($start_date) ELSE null END,
            c.end_date = CASE WHEN $end_date IS NOT NULL THEN date($end_date) ELSE null END,
            c.times = $times,
            c.signup_start = CASE WHEN $signup_start IS NOT NULL THEN localdatetime($signup_start) ELSE null END,
            c.signup_end = CASE WHEN $signup_end IS NOT NULL THEN localdatetime($signup_end) ELSE null END,
            c.spots_available = $spots_available,
            c.languages = $languages,
            c.description = $description,
            c.updated_at = datetime()
        RETURN c
        """
        
        result = session.run(
            query,
            coursecode=course.get("coursecode", ""),
            title=course.get("title", ""),
            link=course.get("link", ""),
            price=course.get("price", ""),
            teacher=course.get("teacher", ""),
            location=course.get("location", ""),
            start_date=course.get("start_date", None),
            end_date=course.get("end_date", None),
            times=course.get("times", ""),
            signup_start=course.get("signup_start", None),
            signup_end=course.get("signup_end", None),
            signup_times=course.get("signup_times", ""),
            spots_available=course.get("spots_available", ""),
            languages=course.get("languages", ""),
            description=course.get("description", "")
        )
        
        return result.single()

created_count = 0
print("Creating Course nodes in Neo4j...")
for course in courses_data:
    # Skip courses that might have failed to scrape proper details
    if not course.get("coursecode"):
        continue
    try:
        result = create_or_update_course_node(driver, course)
        if result:
            created_count += 1
            print(f"✓ Created/Updated: {course['title'][:50]}")
    except Exception as e:
        print(f"✗ Error creating course node: {e}")
        print(f"  Item: {course['title']}")

print(f"\nTotal items processed: {len(courses_data)}")
print(f"Successfully created/updated: {created_count}")

Creating Course nodes in Neo4j...
✓ Created/Updated: After work gymnastik
✓ Created/Updated: Aktiv närkultur
✓ Created/Updated: Aktuellt för pensionärer
✓ Created/Updated: Aktuellt för pensionärer
✓ Created/Updated: Aktuellt för pensionärer
✓ Created/Updated: Animationsfabriken (åk 3-6)
✓ Created/Updated: Att beskära äppelträd
✓ Created/Updated: Bildkonstskolan, fortsättning och fördjupning
✓ Created/Updated: Bildkonstskolan, grundkurs
✓ Created/Updated: Bildkonstskolan, grundkurs
✓ Created/Updated: Bugg inför sommaren
✓ Created/Updated: Cirkelträning i gympasal
✓ Created/Updated: Dans som motion
✓ Created/Updated: Effektiv cirkelträning
✓ Created/Updated: Effektiv cirkelträning - vår/sommar
✓ Created/Updated: Español 2
✓ Created/Updated: FRK Första Hjälpen-kurs FHJ 1® (16 h), kombination
✓ Created/Updated: Flitiga fingrar
✓ Created/Updated: Frisk och skadefri löpare, workshop
✓ Created/Updated: Cirkelträning i gympasal, vår/sommar
✓ Created/Updated: Fräscha upp din engelska
✓ Created/

# **5. Verify and Close Database**

In [31]:
def get_all_course_nodes(driver):
    query = """
    MATCH (c:Course)
    RETURN c.coursecode as code, c.title as title, c.teacher as teacher, c.location as location, c.start_date as start_date, c.end_date as end_date, c.signup_start as signup_start, c.signup_end as signup_end
    ORDER BY c.title ASC
    """
    with driver.session() as session:
        results = session.run(query)
        return [dict(record) for record in results]

all_courses = get_all_course_nodes(driver)
print(f"\n📚 All Course Nodes in Database ({len(all_courses)} total):\n")
for i, c in enumerate(all_courses[:10], 1): # list first 10
    print(f"{i}. {c['title']} (Code: {c['code']})")
    print(f"   Dates: {c['start_date']} to {c['end_date']}")
    print(f"   Signup: {c['signup_start']} to {c['signup_end']}")
    print(f"   Teacher: {c['teacher']}")
    print(f"   Location: {c['location']}")

driver.close()
print("\n✅ Database connection closed successfully!")


📚 All Course Nodes in Database (135 total):

1. After work gymnastik (Code: 830240)
   Dates: 2026-01-12 to 2026-04-13
   Signup: 2026-01-07T08:00:00.000000000 to 2026-04-13T23:59:59.000000000
   Teacher: Linnéa Finne
   Location: Malax - Högstadiet i Petalax, gymnastiksalen Mamrevägen 9, 66240 Petalax
2. Aktiv närkultur (Code: 130100)
   Dates: 2026-01-13 to 2026-04-07
   Signup: 2026-01-07T08:00:00.000000000 to 2026-04-07T23:59:59.000000000
   Teacher: Margareta Björkholm
   Location: Malax - Jossinas stugon 66240 Petalax
3. Aktuellt för pensionärer (Code: 320204)
   Dates: 2026-01-20 to 2026-04-14
   Signup: None to None
   Teacher: Kjell Lolax, Bertel Nygård, Mikael Herrgård, Mia Jåfs, Hans Hästbacka
   Location: Malax - Allaktivitetshuset, Övermalax Kvarnvägen 5 B, 66140 Övermalax
4. Aktuellt för pensionärer (Code: 320208)
   Dates: 2026-01-29 to 2026-04-09
   Signup: None to None
   Teacher: Carina Nordman-Byskata, Janne Sjöström, Sonja Österholm-Granqvist, Benita Sandström-Niem